# Climate data download
<br>
<img style="float: left; padding-right: 15px; padding-left: 0px;" src="../sources/images/logo_continuum.png" width="260px" align=”left” >

<div style="text-align: justify">
This notebook showcases how to download climate data to force the hydrological model Continuum. In particular it will connect to the Copernicus Data Store (CDS) to download hourly Precipitation, Temperature, Solar Radiation, Wind Speed and Dewpoint Temperature from ERA5. Additionally it will show how to download daily precipitation data from CHIRPS v3.


Where text mention **USER ACTION**, you are expected to update paths, names or flags
to match your specific project. Otherwise, you can usually run the cells in sequence
without modifying the code.


## How to use the CDS-API

Import the necessary modules and set up a vew important variables.

**USER_INPUT**

The `cds_api_key` is the key given when you create an ECMWF account.

The `request` you can generate from the [online portal](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download)

In [15]:
import cdsapi
import os

# CDS API key
cds_api_key = 'set your api key here'  # Replace with your actual CDS API key

# Data request parameters
# This will download the 2m temperature data for Ethiopia for October 2025
request = {
    "product_type": ["reanalysis"],
    "variable": ["2m_temperature"],
    "year": ["2025"],
    "month": ["10"],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "area": [8, 37, 10, 40],  # This is a subset of Ethiopia (Addis Ababa), it is smaller to make it faster
    "data_format": "netcdf",
    "download_format": "unarchived"
}

# set and create the output folder
output_folder = os.getcwd().replace('notebook_tools','projects/climate_data_download-test')
os.makedirs(output_folder, exist_ok=True)

# set the output file name
output_file_name = 'era5_2m_temperature_2025_10_ethiopia_raw.nc'
output_file = os.path.join(output_folder, output_file_name)

To download the data, create an instance of the API_client and call it with the request

In [ ]:
# create the CDS API client
c = cdsapi.Client(url = 'https://cds.climate.copernicus.eu/api', key = cds_api_key)

# download the data
c.retrieve('reanalysis-era5-single-levels', request, output_file)

Open the file we downloaded

In [ ]:
# import the xarray module to open the netCDF file
import xarray as xr

# Open the file we downloaded
raw_data = xr.open_dataarray(output_file, engine = 'h5netcdf')

# Display the dataset
print(raw_data)

Clean up the data to put it in the format we need.

In [ ]:
# create a new DataArray with dimensions time, latitude, longitude
data = xr.DataArray(
    raw_data.values,
    coords={
        'time': raw_data['valid_time'].values,
        'latitude': raw_data['latitude'].values,
        'longitude': raw_data['longitude'].values,
    },
    dims=['time', 'latitude', 'longitude'],
    name=raw_data.name,
    attrs={'long_name' : raw_data.attrs.get('long_name'),
           'units' :     raw_data.attrs.get('units')}
)

print(data)

# save the cleaned up data
data.to_netcdf(output_file.replace('_raw',''), engine = 'h5netcdf')

We can also define a few funcions to do all of this automatically for us.

We have created a function to do so in `sources/libraries/download_tools.py`

**DO NOT RUN** the cell below, it will take a lot of time to download all the data

In [ ]:
### DO NOT RUNT THIS CELL AS IT WILL TAKE A LONG TIME AND DOWNLOAD A LOT OF DATA

# import the download function from our library
from download_tools import download_era5_data

# set the output file name
output_file_name = 'ETH_{var}_{year}.nc'
output_file = os.path.join(os.getcwd().replace('notebook_tools','climate_data_download-test'), 'ERA5', '{year}', output_file_name)

# set the extent
extent = [16.00, 31.45, 2.00, 50.50]  # This is Ethiopia

# set the year to download (the data is downloaded in 1 year chunks)
year = 2020

# Download ERA5 data
download_era5_data(year, bbox=extent, output=output_file, key=cds_api_key)

## How to download CHIRPS v3 data (programmatically)

To download CHIRPS data, we can use the `requests` package to directly download the file from its `http` address.

In [ ]:
# import the dependencies
import requests
import rioxarray as rxr

# this it the CHIRPS data url sample
url_blank = 'https://data.chc.ucsb.edu/products/CHIRPS/v3.0/daily/final/rnl/{year}/chirps-v3.0.rnl.{year}.{month:02d}.{day:02d}.tif'

# choose a date to download
year = 2025
month = 10
day = 1
# set the extent
extent = [16.00, 31.45, 2.00, 50.50]  # This is Ethiopia

# set the output file name
output_file_name = f'chirps-precip_{year}{month:02d}{day:02d}_raw.tif'
output_file = os.path.join(os.getcwd().replace('notebook_tools','climate_data_download-test'), output_file_name)

# format the url
url = url_blank.format(year = year, month = month, day = day)
print(url)

response = requests.get(url)
# download the file
if response.status_code == 200: # 200 means the url is valid
    with open(output_file, 'wb') as f:
        f.write(response.content)
else:
    raise ValueError(f"CHIRPS URL not found: {url}")

# open the file to check it
data_raw = rxr.open_rasterio(output_file)
print(data_raw)

Clean up the data to put it in the format we need.

In [14]:
# set crs and coordinates
data_raw = data_raw.rio.write_crs("EPSG:4326")
data_raw = data_raw.rio.set_spatial_dims(x_dim="x", y_dim="y")

# subset the data to the bounding box
subset_data = data_raw.rio.clip_box(*(extent[1:] + extent[:1]))  # West, South, East, North

# ensure the data is straight-up
subset_data = subset_data.sortby('y', ascending=False)

# set nans (nans in chirps are -9999, we need nan instead)
subset_data = subset_data.where(subset_data >= 0, other = float('nan'))
subset_data.attrs['_FillValue'] = float('nan')

# set attributes
subset_data.name = 'CHIRPS_daily_precipitation'
subset_data.attrs['long_name'] = 'CHIRPS Daily Precipitation'
subset_data.attrs['units'] = 'mm/day'

# set the output file name
output_file_name = f'chirps-precip_{year}{month:02d}{day:02d}.tif'
output_file = os.path.join(os.getcwd().replace('notebook_tools','climate_data_download-test'), output_file_name)

# save the file
subset_data.rio.to_raster(output_file, compress = 'LWZ')

Also in this case, we can also define a few funcions to do all of this automatically for us.

We have created a function to do so in `sources/libraries/download_tools.py`

**DO NOT RUN** the cell below, it will take a lot of time to download all the data

In [ ]:
### DO NOT RUNT THIS CELL AS IT WILL TAKE A LONG TIME AND DOWNLOAD A LOT OF DATA

# import the download function from our library
from download_tools import download_chirps_data

# set the output file name
output_file_name = 'CHIRPS-precip1d_{year}{month:02d}{day:02d}.tif'
output_file = os.path.join(os.getcwd().replace('notebook_tools','climate_data_download-test'), 'CHIRPS', '{year}', output_file_name)

# set the extent
extent = [16.00, 31.45, 2.00, 50.50]  # This is Ethiopia

# set the year to download (the data is downloaded in 1 year chunks)
year = 2020

# Download ERA5 data
download_chirps_data(year, bbox=extent, output=output_file)